# IAN 620 — LLM Tool Calling with LangChain and LangGraph

**Goal:** Ask a natural-language question. Qwen decides which tool(s) to call based on tool **names, docstrings, and argument schemas**. LangGraph runs the selected tools and sends results back to the model.

Three example tools:
- **Weather**: API integration is commented out; returns clearly labeled sample weather.
- **Jobs**: API integration is commented out; returns fictional sample jobs.
- **Profit**: performs a real calculation from the numbers in the question.

A combined request may call multiple tools. There is **no Python keyword-based router, custom JSON planner, or forced tool selection**.

**Google Colab:** Runtime → Change runtime type → **T4 GPU** (recommended). Run the notebook top to bottom. Ollama and the model download require internet access. No paid model API key is needed.

## 1. Install libraries and start Ollama (Colab)

Some recent Linux runtimes do not include **zstd**, which Ollama's installer requires for unpacking its release. We install that prerequisite first, then download the installation script **separately** so that failed downloads are visible. <https://docs.ollama.com/linux>

If the command fails, the error is displayed rather than hidden by the original `curl | sh` pipeline. Rerun this cell only after fixing the displayed error.

Recommended: Runtime → Change runtime type → T4 GPU.

In [ ]:
%pip -q install -U langchain langgraph langchain-ollama requests

import shutil
import subprocess
import time
from pathlib import Path

import requests

def run_command(command):
    """Run a shell command and display its actual error if it fails."""
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

if shutil.which("ollama") is None:
    # The Ollama installer needs zstd on newer Linux distributions.
    run_command(["apt-get", "update", "-qq"])
    run_command(["apt-get", "install", "-y", "-qq", "zstd", "curl", "ca-certificates"])

    # Download first (rather than curl | sh) to expose download errors.
    install_script = Path("/tmp/ollama_install.sh")
    response = requests.get("https://ollama.com/install.sh", timeout=90)
    response.raise_for_status()
    install_script.write_bytes(response.content)

    # Check the actual installer return code; show its output on failure.
    run_command(["bash", str(install_script)])

run_command(["ollama", "--version"])

# Ollama runs as a local HTTP service for subsequent cells.
try:
    requests.get("http://127.0.0.1:11434/api/tags", timeout=3).raise_for_status()
except requests.RequestException:
    log = open("/tmp/ollama-server.log", "a")
    server = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log,
        stderr=subprocess.STDOUT,
    )
    for _ in range(45):
        try:
            response = requests.get("http://127.0.0.1:11434/api/tags", timeout=3)
            if response.ok:
                break
        except requests.RequestException:
            pass
        time.sleep(1)
    else:
        log.flush()
        print(Path("/tmp/ollama-server.log").read_text()[-3000:])
        raise RuntimeError("Ollama did not start; inspect the server log above.")

print("Ollama API ready.")

## 2. Download and connect a tool-calling Qwen model

This tutorial uses **Qwen 2.5 (3B)**. It is small enough for a typical Colab GPU runtime and supports tool-calling through Ollama. A larger model may choose tools more reliably.

The model is responsible for deciding **whether** to use a tool, **which** tool to use, and **what arguments** to supply.

In [ ]:
from langchain_ollama import ChatOllama

MODEL_NAME = "qwen2.5:3b"
subprocess.run(["ollama", "pull", MODEL_NAME], check=True)

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0,
    num_ctx=4096,
)
print("Model connected:", MODEL_NAME)

## 3. Define three LangChain tools

A function's **docstring becomes its tool description**. The LLM sees the tool names, descriptions, and argument schemas when we call `bind_tools`.

**The docstring describes the tool. It does not run it.** LangGraph executes the chosen tools later. The weather and jobs API calls are intentionally commented out; do not interpret their examples as current information.

In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get current weather information for a city.

    Use for requests about current conditions, temperature, rain, or forecasts.
    The city argument should be the location requested by the user.
    """
    # Real API example (requires a weather API provider and key):
    # response = requests.get(WEATHER_API_URL, params={"city": city, "key": API_KEY}, timeout=15)
    # response.raise_for_status()
    # return str(response.json())
    return f"DEMO WEATHER, not live data: {city}: 72 F, partly cloudy."


@tool
def search_jobs(city: str) -> str:
    """Find recent job opportunities in a specified city.

    Use for requests about hiring, openings, employment, or available jobs.
    The city argument should be the location requested by the user.
    """
    # Real API example (requires an authorized jobs API provider):
    # response = requests.get(JOBS_API_URL, params={"location": city}, timeout=15)
    # response.raise_for_status()
    # return str(response.json())
    return (
        f"FICTIONAL JOB EXAMPLES ONLY — not verified openings in {city}:\n"
        "1. Data Analyst — Example Analytics Co.\n"
        "2. Python Developer — Demo Software Inc.\n"
        "3. AI Engineer — Sample Technology Group"
    )


@tool
def calculate_profit(selling_price: float, cost_price: float, quantity: int) -> str:
    """Calculate total profit and profit margin for products.

    Use when asked to compute business profit or earnings from selling price,
    cost per item, and quantity. Pass the actual numeric inputs provided by
    the user; if a required value is missing, ask for it instead of guessing.
    """
    if quantity < 0:
        return "Error: quantity must not be negative."
    revenue = selling_price * quantity
    cost = cost_price * quantity
    profit = revenue - cost
    margin = (profit / revenue * 100) if revenue else 0
    return (
        f"Revenue: ${revenue:,.2f}; Total cost: ${cost:,.2f}; "
        f"Profit: ${profit:,.2f}; Margin: {margin:.1f}%."
    )


tools = [get_weather, search_jobs, calculate_profit]

for item in tools:
    print(item.name, "—", item.description.splitlines()[0])

## 4. Bind the tools to the language model

`bind_tools(tools)` makes tool schemas available to Qwen. **No agent is selected in Python.**

The result is an `AIMessage`. When the model wants to call a tool, its `tool_calls` attribute contains the requested function name and arguments. The model can also answer without a tool.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = """You are a helpful assistant.
Use available tools when information or exact calculation is needed.
Choose tools from their descriptions and the user's actual request.
For requests requiring several tools, call all relevant tools.
Never invent a missing city, price, or quantity; ask for clarification.
Weather and jobs tools currently return SAMPLE data. Clearly label them
as demos, never as live weather or actual current vacancies.
For profit calculations, use calculate_profit instead of mental math.
If no tool is relevant, answer directly.
After a tool returns, summarize its answer without calling the same
tool again unless the user genuinely needs a new request."""

preview = llm_with_tools.invoke([
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content="What is the temperature in Greensboro?"),
])

print("Tool calls proposed by the LLM:")
print(preview.tool_calls)
print("Other response:", preview.content)

## 5. Build the agent workflow with LangGraph

The diagram is:

**START → LLM → (tool calls?) → ToolNode → LLM → END**

The decision is the LLM's. The `tools_condition` function only checks whether the model actually requested a tool. It does **not** look for words such as "weather" or "jobs" in the question.

`ToolNode` executes tool calls automatically; the results are returned to the model as `ToolMessage` objects. The model can then write the final answer.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def call_model(state: MessagesState):
    response = llm_with_tools.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        *state["messages"],
    ])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("llm", call_model)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "llm")
builder.add_conditional_edges(
    "llm", tools_condition,
    {"tools": "tools", "__end__": END},
)
builder.add_edge("tools", "llm")

assistant = builder.compile()
print("LangGraph agent ready.")

## 6. Ask questions and inspect which tools were called

The model decides tool selection. You can inspect the actual calls instead of assuming a fixed route. We set a graph recursion limit to avoid endlessly repeated model/tool steps.

In [ ]:
def ask(question: str) -> None:
    print("\nUSER:", question)
    result = assistant.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"recursion_limit": 12},
    )

    for message in result["messages"]:
        if getattr(message, "tool_calls", None):
            for call in message.tool_calls:
                print("LLM SELECTED TOOL:", call["name"])
                print("ARGUMENTS:", call["args"])
        if message.type == "tool":
            print("TOOL RESULT:", message.content)
    print("FINAL ANSWER:", result["messages"][-1].content)
    print("-" * 60)

ask("How warm is it in Greensboro?")

## 7. Three example requests

The **jobs and weather results are fictional/sample outputs**, while the profit result uses real arithmetic. Try rephrasing the requests to see whether the model picks tools based on their descriptions.

In [ ]:
ask("Are any companies hiring in Greensboro?")
ask("I buy 10 mugs for 50 dollars each and sell them for 75 each. What is my total profit?")

## 8. One question, multiple tools

This request requires two different sources. Qwen should propose tool calls for both. In the printed trace, look for **both** `search_jobs` and `get_weather`.

Note: Small models can still make incorrect selections. Tool-call accuracy is a model capability, not something the docstring guarantees.

In [ ]:
ask("Are there any jobs in Greensboro, and what is the weather there?")

## 9. Try your own question

There are no manual routing rules. Change the `question` variable and run the cell.

In [ ]:
question = "I sell 4 items for $120 each after buying each for $85. What do I earn?"
ask(question)

## 10. Optional: display the graph

This graph illustrates the model → tools → model loop. It is **not** a hardcoded weather/jobs/profit decision tree.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(assistant.get_graph().draw_mermaid_png()))
except Exception:
    print(assistant.get_graph().draw_mermaid())

## 11. What to learn from this example

- **`@tool`** wraps a Python function and exposes its name, docstring, and parameters.
- **`bind_tools`** lets a tool-capable language model decide which functions to request.
- **`AIMessage.tool_calls`** shows the names and arguments requested by the model.
- **`ToolNode`** runs the tools, while **`tools_condition`** routes based on the presence of actual tool calls.
- **LangGraph** sends the observations back to the model so it can finish its answer.

**Try:** Change a tool's docstring. Does the model choose tools differently? Ask a question requiring two tools. Ask one that requires no tool. Ask for profit without a quantity and observe whether the model asks for clarification.

**Next steps:** Add real APIs with keys stored in Colab Secrets, HTTP timeouts, response validation, error handling, and a stronger LLM if tool calls are unreliable.

Resources: [LangChain tool calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling) · [LangGraph ToolNode](https://reference.langchain.com/python/langgraph.prebuilt/tool_node/ToolNode) · [ChatOllama](https://docs.langchain.com/oss/python/integrations/chat/ollama)